# Suite VLGRD — Reading Verilog with slang

The standards read source text through slang, a complete SystemVerilog front end that runs in Python: each read runs
the Python implementation's reader and loads the tree it writes, so both implementations read the same trees. The
conformance source reads to a valid tree that prints and reads back the same; each construct reads to its kinds;
directives, comments and macro uses are kept where items and statements are listed; and what slang reports, the reader
refuses or a standard lacks fails with its line and column.

In [ ]:
import { mkdtempSync, readFileSync, rmSync, writeFileSync } from "node:fs";
import { tmpdir } from "node:os";
import { join } from "node:path";

import { Errors, Syntax as F } from "@mbse/programs/Framework";
import { SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard } from "@mbse/programs/Verilog";
import * as Reader from "@mbse/programs/Verilog/_Reader";

import { assert, equal, raises } from "./support.js";

const { ParseError } = Errors;
const L = S.LANGUAGE;
const SOURCES = "../../conformance/sources";
const read = (text: string, includePaths: string[] = [], defines: string[] = []) =>
  SystemVerilog2023.parse(text, includePaths, defines);
const items = (text: string): any[] => (read(`module m;\n${text}\nendmodule\n`).items[0] as any).items;
const statements = (text: string): any[] => items(`initial begin\n${text}\nend`)[0].body.items;
const expression = (text: string): any => statements(`x = ${text};`)[0].value;
const kind = (node: any) => node.kind().KIND;
function fails(text: string, message: string, includePaths: string[] = []): void {
  raises(ParseError, () => read(text, includePaths), message);
}
let unit: any, module: any, body: any[], declared: any[], conditional: any, loop: any;

## VLGRD-01 · The conformance source reads to a valid tree that prints and reads back the same

In [ ]:
const text = readFileSync(`${SOURCES}/systemverilog2023.sv`, "utf8");
const tree = read(text, [SOURCES]);
equal(L.validate(tree), []);
equal(new Set([...F.walk(tree)].map(kind)), new Set(L.kinds().keys()));
const printed = SystemVerilog2023.print(tree);
const again = read(printed, [SOURCES]);
assert(F.same(again, tree) && SystemVerilog2023.print(again) === printed);
console.log([...F.walk(tree)].length, "syntax nodes");

## VLGRD-02 · Design units: headers, imports, parameters, ANSI, interface and non-ANSI ports

In [ ]:
unit = read(`package p; endpackage
module automatic m import p::*; #(parameter int W = 8, localparam N = 2, parameter type T = logic) (
    input wire [W-1:0] a, output var logic b = 1'b0, inout tri c, ref int r, bus.mp port, interface generic [2]);
endmodule : m
macromodule n (a, b); input a; output reg [3:0] b; endmodule
interface automatic i; endinterface
program static q; endprogram
`);
const m = unit.items[1];
equal([m.keyword, m.lifetime, m.name.spelling, m.labeled], ["module", "automatic", "m", true]);
assert(m.imports[0].items[0].package.spelling === "p" && m.imports[0].items[0].name === null);
const [parameter, local, typeParameter] = m.parameters;
equal([parameter.keyword, parameter.type.keyword, parameter.assignments[0].name.spelling], ["parameter", "int", "W"]);
assert(local.keyword === "localparam" && local.type === null && local.assignments[0].value.spelling === "2");
assert(typeParameter instanceof S.TypeParameterDeclaration && (typeParameter.assignments[0]?.type as any).keyword === "logic");
const [a, b, c, r, port, generic] = m.ports;
equal([a.direction, a.net_type, a.type.dimensions[0].left.left.name.spelling], ["input", "wire", "W"]);
equal([b.direction, b.var, b.type.keyword, b.value.spelling], ["output", true, "logic", "1'b0"]);
equal([c.net_type, r.direction, r.type.keyword], ["tri", "ref", "int"]);
assert(port instanceof S.InterfacePort);
equal([port.interface.spelling, port.modport.spelling], ["bus", "mp"]);
assert(generic.interface === null && generic.modport === null && generic.dimensions[0].size.spelling === "2");
const n = unit.items[2];
assert(n.keyword === "macromodule");
equal(n.ports.map((p: any) => p.name.spelling), ["a", "b"]);
assert(n.items[0] instanceof S.PortDeclaration && n.items[0].declarators[0].name.spelling === "a");
assert(n.items[1].type.keyword === "reg" && n.items[1].direction === "output");
assert(unit.items[3].lifetime === "automatic" && unit.items[4].lifetime === "static");
const bare: any[] = (read("module k #(int W = 8, type T = logic) (); endmodule\n").items[0] as any).parameters;
assert(bare[0].keyword === null && bare[0].type.keyword === "int" && bare[1].keyword === null);
equal(L.validate(bare[0]), []);
console.log("ok");

## VLGRD-03 · Data types and dimensions

In [ ]:
declared = items(`logic signed [7:0] a [4][0:1];
int unsigned b [string];
byte c [*];
shortreal d [$:8];
string e [];
p::t f;
struct packed signed { bit [3:0] x, y; } g;
union { int i; real r; } h;
enum { A, B = 2 } k;
event l;
realtime m;
chandle n;`);
const logic = declared[0];
equal([logic.type.keyword, logic.type.signing, logic.type.dimensions[0].right.spelling], ["logic", "signed", "0"]);
assert(logic.declarators[0].dimensions[0].size.spelling === "4" && logic.declarators[0].dimensions[1] instanceof S.RangeDimension);
assert(declared[1].type.signing === "unsigned" && declared[1].declarators[0].dimensions[0].type.keyword === "string");
assert(declared[2].declarators[0].dimensions[0].type === null);
assert(declared[3].type.keyword === "shortreal" && declared[3].declarators[0].dimensions[0].bound.spelling === "8");
assert(declared[4].declarators[0].dimensions[0] instanceof S.UnsizedDimension && declared[4].type.keyword === "string");
equal([declared[5].type.name.scope.spelling, declared[5].type.name.name.spelling], ["p", "t"]);
const g = declared[6].type;
equal([g.keyword, g.packed, g.signing, g.members[0].declarators.map((d: any) => d.name.spelling)],
  ["struct", true, "signed", ["x", "y"]]);
assert(declared[7].type.keyword === "union" && !declared[7].type.packed);
const k = declared[8].type;
assert(k.base === null && k.members[1].value.spelling === "2");
equal(k.members.map((m: any) => m.name.spelling), ["A", "B"]);
equal(declared.slice(9).map((d) => d.type.keyword), ["event", "realtime", "chandle"]);
console.log("ok");

## VLGRD-04 · Declarations and items: nets, variables, typedefs, genvars, imports, modports, assigns and constructs

In [ ]:
declared = items(`wire [1:0] #3 w = 2'b01, v;
const var static int x = 1;
typedef logic [3:0] nibble_t [2];
genvar g, h;
import p::a, p::*;
assign #1 w = v, v = 0;
always @* x = 1;
always_comb x = 2;
always_latch x = 3;
initial x = 4;
final x = 5;
function automatic void f(); endfunction : f
function g2(input a); return a; endfunction
task t(output int o, input logic [1:0] i = 2'b0); endtask
task old; input a; endtask`);
const [wire, variable, typedef, genvar, imports, assign] = declared;
equal([wire.net_type, wire.delay.value.spelling, wire.type.dimensions[0].left.spelling], ["wire", "3", "1"]);
equal(wire.declarators.map((d: any) => d.name.spelling), ["w", "v"]);
assert(wire.declarators[0].value.spelling === "2'b01");
equal([variable.const, variable.var, variable.lifetime], [true, true, "static"]);
assert(typedef.dimensions[0].size.spelling === "2");
equal(genvar.names.map((n: any) => n.spelling), ["g", "h"]);
equal(imports.items.map((i: any) => [i.package.spelling, i.name && i.name.spelling]), [["p", "a"], ["p", null]]);
assert(assign.delay.value.spelling === "1" && assign.assignments.length === 2);
equal(declared.slice(6, 9).map((c) => c.keyword), ["always", "always_comb", "always_latch"]);
assert(declared[9] instanceof S.InitialConstruct && declared[10] instanceof S.FinalConstruct);
const [f, g2, t, old] = declared.slice(11);
equal([f.lifetime, f.type.keyword, f.labeled, f.ports], ["automatic", "void", true, []]);
assert(g2.type === null && g2.ports[0].direction === "input" && g2.body[0] instanceof S.ReturnStatement);
assert(t.ports[1].value.spelling === "2'b0" && t.ports[0].type.keyword === "int");
assert(old.ports.length === 0 && old.body[0] instanceof S.PortDeclaration);
const iface: any = read("interface i; modport mp (input a, output b, inout c); endinterface").items[0];
equal(iface.items[0].items[0].ports.map((p: any) => [p.direction, p.name.spelling]), [["input", "a"], ["output", "b"],
  ["inout", "c"]]);
console.log("ok");

## VLGRD-05 · Generate constructs and instantiation

In [ ]:
declared = items(`generate for (genvar i = 0; i < 2; i++) begin : g end endgenerate
for (j = 0; j < 2; j = j + 1) assign a[j] = 0;
if (W) begin end else if (V) assign a = 1; else begin : e end : e
case (W) 1, 2: assign a = 1; default: begin end endcase
sub #(1, .N(2), .T(logic)) u1 (a, .b(c), .d(), .e), u2[1:0] (.*);`);
loop = declared[0].items[0];
assert(loop.genvar && loop.name.spelling === "i" && loop.step instanceof S.IncrementExpression && loop.body.name.spelling === "g");
assert(!declared[1].genvar && declared[1].body instanceof S.ContinuousAssign);
conditional = declared[2];
assert(conditional.alternative instanceof S.GenerateIf && conditional.alternative.alternative.labeled);
equal(declared[3].items.map((i: any) => i.expressions.length), [2, 0]);
const instantiation = declared[4];
equal(instantiation.parameters.map(kind), ["IntegerLiteral", "NamedConnection", "NamedConnection"]);
assert(instantiation.parameters[2].value.keyword === "logic");
const [u1, u2] = instantiation.instances;
equal(u1.connections.map(kind), ["NameExpression", "NamedConnection", "NamedConnection", "NamedConnection"]);
assert(u1.connections[2].value === null && !u1.connections[2].implicit && u1.connections[3].implicit);
assert(u2.connections[0] instanceof S.WildcardConnection && u2.dimensions[0].left.spelling === "1");
console.log("ok");

## VLGRD-06 · Statements

In [ ]:
body = statements(`a = 1; b <= #2 c; d += 1; e <= @(posedge clk) f;
f(); i++; --j; ;
begin : named int k; end : named
fork join_any
fork : par join_none
unique if (a) b = 1; else if (c) b = 2; else b = 3;
priority casez (a) inside 1, [2:3]: ; default b = 0; endcase
casex (a) 1: ; endcase
for (int k = 0, m = 1; k < 2; k++, m--) ;
for (;;) break;
while (a) continue;
repeat (2) ;
forever ;
do a = 1; while (b);
foreach (q[i, j]) a = 1;
#1 ;
@(a or posedge b, negedge c iff d) ;
@e a = 1;
@(*) ;
wait (a) ;
-> ev;
->> ev;
disable named;
disable fork;
assert (a) b = 1; else b = 2;
assume #0 (a);
cover final (a) b = 1;
return;`);
equal([body[0].operator, body[0].timing], ["=", null]);
assert(body[1].operator === "<=" && body[1].timing.value.spelling === "2");
assert(body[2].operator === "+=" && body[3].timing.events[0].edge === "posedge");
assert(body[4].expression instanceof S.CallExpression && body[5].expression.postfix && !body[6].expression.postfix);
assert(body[7] instanceof S.NullStatement);
const named = body[8];
assert(named.name.spelling === "named" && named.labeled && named.items[0] instanceof S.VariableDeclaration);
assert(body[9].join === "join_any" && body[10].name.spelling === "par" && body[10].join === "join_none");
conditional = body[11];
assert(conditional.qualifier === "unique" && conditional.alternative.qualifier === null);
assert(conditional.alternative.alternative instanceof S.AssignmentStatement);
const caseStatement = body[12];
equal([caseStatement.qualifier, caseStatement.keyword, caseStatement.inside], ["priority", "casez", true]);
assert(caseStatement.items[0].expressions[1] instanceof S.ValueRange && caseStatement.items[1].expressions.length === 0);
assert(body[13].keyword === "casex");
loop = body[14];
equal(loop.initializers.map(kind), ["VariableDeclaration", "VariableDeclaration"]);
assert(loop.steps.length === 2);
assert(loop.initializers[0].type.keyword === "int" && loop.initializers[1].type === null); // `m` is an int too
assert(body[15].condition === null && body[15].initializers.length === 0);
equal(body.slice(16, 21).map(kind), ["WhileStatement", "RepeatStatement", "ForeverStatement", "DoWhileStatement",
  "ForeachStatement"]);
equal(body[20].variables.map((v: any) => v.spelling), ["i", "j"]);
assert(body[21].body === null && body[21].timing.value.spelling === "1");
const events = body[22].timing.events;
equal(events.map((e: any) => [e.edge, e.expression.name.spelling]), [[null, "a"], ["posedge", "b"], ["negedge", "c"]]);
assert(events[2].condition.name.spelling === "d");
assert(body[23].timing.events[0].expression.name.spelling === "e" && body[24].timing.events.length === 0);
assert(body[25].body === null && !body[26].nonblocking && body[27].nonblocking);
assert(body[28].target.name.spelling === "named" && body[29].target === null);
const asserted = body[30];
assert(asserted.pass_action.operator === "=" && asserted.fail_action.value.spelling === "2");
assert(body[31].deferral === "#0" && body[31].keyword === "assume" && body[32].deferral === "final");
assert(body[33].value === null);
console.log("ok");

## VLGRD-07 · Expressions

In [ ]:
const values = ["a + b * c", "(a + b) * c", "-a", "&a", "~^a", "a ? b : c", "a inside {1, [2:3]}", "{a, b}", "{2{a}}",
  "'{1, 2}", "'{default: 0, a: 1, int: 2}", "t'{1}", "f(1, .b(2))", "$clog2(1)", "$time", "q.size()",
  "int'(x)", "8'(x)", "signed'(x)", "p::c", "s.m[1][7:0]", "a[i +: 2]", "a[i -: 2]", "q[$]", "8'hFF", "1.5e3",
  "10ns", "'1", "\"s\\n\"", "x++", "(y = 1)", "a <-> b"].map(expression);
assert(values[0].operator === "+" && values[0].right.operator === "*");
assert(values[1].left instanceof S.ParenthesizedExpression);
equal(values.slice(2, 5).map((v) => v.operator), ["-", "&", "~^"]);
assert(values[5] instanceof S.ConditionalExpression && values[6].set[1] instanceof S.ValueRange);
assert(values[7].items.length === 2 && values[8].count.spelling === "2");
assert(values[9].items[1].spelling === "2" && values[9].type === null);
equal(values[10].items.map((i: any) => i.key === null ? null : kind(i.key)), [null, "NameExpression", "IntegerAtomType"]);
assert(values[11].type.name.spelling === "t");
assert(values[12].arguments[1] instanceof S.NamedConnection && values[13].name === "$clog2");
assert(values[14].arguments.length === 0 && values[15].callee.member.spelling === "size");
assert(values[16].type.keyword === "int" && values[17].type.spelling === "8" && values[18].type.signing === "signed");
assert(values[19].name.scope.spelling === "p");
const member = values[20];
assert(member instanceof S.RangeSelect && member.value instanceof S.IndexExpression);
assert((member.value.value as any).member.spelling === "m");
assert(values[21].operator === "+:" && values[22].operator === "-:" && values[23].index instanceof S.DollarExpression);
equal(values.slice(24, 29).map(kind), ["IntegerLiteral", "RealLiteral", "TimeLiteral", "UnbasedUnsizedLiteral", "StringLiteral"]);
assert(values[24].spelling === "8'hFF" && values[27].value === "1" && values[28].text === "s\\n");
assert(values[29].postfix && values[30].expression instanceof S.AssignmentExpression && values[31].operator === "<->");
console.log("ok");

## VLGRD-08 · Directives and comments where items and statements are listed; conditional compilation as a tree

In [ ]:
unit = read(`// one
\`resetall
\`define A 1
\`define F(x, y) ((x) + (y))
\`undef A
\`default_nettype wire
\`timescale 10ns / 1ns
\`ifndef B
module m; /* inner */
\`ifdef C
wire c;
\`elsif D
wire d;
\`else
wire e; // trailing
\`endif
initial begin
\`ifdef C
x = 1;
\`endif
end
endmodule
\`endif
`);
const [comment, resetall, define, macro, undef, nettype, timescale, ifndef] = unit.items;
assert(comment.text === " one" && !comment.trailing && resetall.text === "`resetall");
equal([define.name.spelling, define.body, define.function_like], ["A", "1", false]);
assert(macro.function_like);
equal(macro.parameters.map((p: any) => p.spelling), ["x", "y"]);
assert(undef.name.spelling === "A" && nettype.net_type === "wire");
equal([timescale.unit, timescale.precision], ["10ns", "1ns"]);
assert(ifndef.negated && ifndef.name.spelling === "B" && !ifndef.has_else);
module = ifndef.items[0];
const [inner, ifdef] = module.items;
assert(inner.block && inner.trailing);
assert(ifdef.items[0] instanceof S.DisabledText && ifdef.items[0].text === "wire c;");
assert(ifdef.branches[0].items[0] instanceof S.DisabledText && ifdef.has_else);
assert(ifdef.alternative[0] instanceof S.NetDeclaration && ifdef.alternative[1].trailing);
const statementIfdef = module.items[2].body.items[0];
assert(statementIfdef instanceof S.IfdefDirective && (statementIfdef.items[0] as any).text === "x = 1;");
const taken: any = read("`ifdef C\nmodule m; endmodule\n`endif\n", [], ["C"]);
assert(taken.items[0].items[0] instanceof S.ModuleDeclaration);
assert(SystemVerilog2023.print(unit).startsWith("// one\n`resetall\n`define A 1\n`define F(x, y) ((x) + (y))"));
console.log("ok");

## VLGRD-09 · Macro uses and included files

In [ ]:
unit = read(`\`define W 8
\`define SUM(a, b) a + b
module m; wire [\`W-1:0] x = \`SUM(1, 2); endmodule
`);
const declaration = unit.items[2].items[0];
const width = declaration.type.dimensions[0].left.left;
assert(width instanceof S.MacroUsage && width.name?.spelling === "W" && width.arguments === null);
const value = declaration.declarators[0].value;
assert(value instanceof S.MacroUsage && value.arguments === "1, 2");
const folder = mkdtempSync(join(tmpdir(), "verilog-"));
try {
  writeFileSync(join(folder, "inc.svh"), "`define FROM_INCLUDE 1\nwire included;\n");
  const included: any = read("module m;\n`include \"inc.svh\"\nwire own;\nendmodule\n", [folder]);
  equal(included.items[0].items.map(kind), ["IncludeDirective", "NetDeclaration"]);
  assert(included.items[0].items[1].declarators[0].name.spelling === "own");
} finally {
  rmSync(folder, { recursive: true });
}
fails("`include \"missing.svh\"\nmodule m; endmodule\n", "could not open include file");
console.log("ok");

## VLGRD-10 · What slang reports, what the reader does not support and what a standard lacks fail at their place

In [ ]:
fails("module m;\n  wire = ;\nendmodule\n", "line 2");
fails("module m; endmodule\nmodule n; covergroup cg; endgroup endmodule\n", "line 2, column 11: unsupported syntax: CovergroupDeclaration");
fails("`define S a = 1;\nmodule m; initial begin `S end endmodule\n", "a macro use that is not a whole expression");
fails("checker c; endchecker\n", "unsupported syntax: CheckerDeclaration");
fails("module m; wire [3:0] a = 1 + `undefined_macro; endmodule\n", "unknown directive");
raises(ParseError, () => Verilog2005.parse("package p; endpackage\n"),
  "line 1, column 1: PackageDeclaration is not Verilog, but this is Verilog-2005");
assert(Verilog2005.parse("module m (input a); endmodule\n").items[0] instanceof S.ModuleDeclaration);
let error: any = null;
try {
  read("module m;\n\n   wire = ;\nendmodule\n");
} catch (caught) {
  error = caught;
}
assert(error instanceof ParseError);
equal([error.line, error.column], [3, 9]);
console.log("ok");

## VLGRD-11 · What is outside the kinds is refused, naming it, where it is

In [ ]:
for (const [source, message] of [
  ["`define N n\nmodule m; wire `N; endmodule", "line 2, column 16: unsupported syntax: a macro use that is not a whole expression"],
  ["`ifdef (A && B)\nmodule m; endmodule\n`endif", "unsupported syntax: IfDefDirective"],
  ["`ifdef A\nmodule a;\n`else\nmodule b;\n`endif\nendmodule", "line 5, column 1: unexpected `endif"],
  ["module m (.*); endmodule", "unsupported syntax: WildcardPortList"],
  ["module m (a[0]); input [1:0] a; endmodule", "unsupported syntax: ImplicitNonAnsiPort"],
  ["module m (.p(a)); input a; endmodule", "unsupported syntax: ExplicitNonAnsiPort"],
  ["module m #(parameter type enum T = e) (); endmodule", "unsupported syntax: TypeParameterDeclaration"],
  ["module m; typedef struct { rand int a; } t; endmodule", "unsupported syntax: StructUnionMember"],
  ["module m; enum { A[2] } e; endmodule", "unsupported syntax: Declarator"],
  ["module m; wire [3+:2] a; endmodule", "unsupported syntax: AscendingRangeSelect"],
  ["module m; wire (strong0, weak1) a; endmodule", "unsupported syntax: NetDeclaration"],
  ["module m; wire vectored [1:0] a; endmodule", "unsupported syntax: NetDeclaration"],
  ["interface i; modport mp (import f); endinterface", "unsupported syntax: ModportSubroutinePortList"],
  ["interface i; modport mp (input .a(b)); endinterface", "unsupported syntax: ModportExplicitPort"],
  ["module m; assign (strong0, weak1) a = 1; endmodule", "unsupported syntax: ContinuousAssign"],
  ["module m; function int f(const ref int a); endfunction endmodule", "unsupported syntax: FunctionPort"],
  ["module m; u i(a, , b); endmodule", "unsupported syntax: EmptyPortConnection"],
  ["module m; initial if (a matches 1) ; endmodule", "unsupported syntax: ConditionalPredicate"],
  ["module m; initial case (a) matches 1: ; endcase endmodule", "unsupported syntax: CaseStatement"],
  ["module m; initial wait fork; endmodule", "unsupported syntax: WaitForkStatement"],
  ["module m; initial ->> #1 e; endmodule", "unsupported syntax: NonblockingEventTriggerStatement"],
  ["module m; initial #(1:2:3) ; endmodule", "unsupported syntax: MinTypMaxExpression"],
  ["module m; initial x = f(a, ,b); endmodule", "unsupported syntax: EmptyArgument"],
  ["module m; initial x = '{2{a}}; endmodule", "unsupported syntax: ReplicatedAssignmentPattern"],
  ["module m; initial x = {>>{a}}; endmodule", "unsupported syntax: StreamingConcatenationExpression"],
  ["module m; initial x = $root.a; endmodule", "unsupported syntax: RootScope"],
  ["module m; typedef union tagged { int a; } t; endmodule", "unsupported syntax: UnionType"],
  ["module m; int [1:0] a; endmodule", "unsupported syntax: IntType"],
  ["module m; assign #(1,2) a = b; endmodule", "unsupported syntax: Delay3"],
  ["module m; initial #1step ; endmodule", "unsupported syntax: OneStepDelay"],
  ["`pragma protect begin_protected\n`pragma protect data_block\nabc\n`pragma protect end_protected\nmodule m; endmodule",
    "unsupported syntax: skipped text"],
  ["module a;\n`ifdef A\n`else\nendmodule\n`endif", "line 3, column 1: expected `endif"],
  ["module m (a); bus_if.mp a; endmodule", "unsupported syntax: InterfacePortHeader"],
  ["module m; typedef a.b t; endmodule", "unsupported syntax: ScopedName"],
  ["module m; initial foreach (q[, j]) x = 1; endmodule", "unsupported syntax: EmptyIdentifierName"],
  ["module m; initial x = f(a[*2]); endmodule", "unsupported syntax: SimpleSequenceExpr"],
  ["module m; initial x = a.b#(1); endmodule", "unsupported syntax: ScopedName"],
  ["module m; initial x = a inside {[1 +/- 2]}; endmodule", "unsupported syntax: ValueRangeExpression"],
  ["module m; initial x = $unit::a; endmodule", "unsupported syntax: ScopedName"],
  ["module m; (* keep *) wire a; endmodule", "line 1, column 11: unsupported syntax: AttributeInstance"],
  ["class :final c; endclass", "unsupported syntax: ClassDeclaration"],
  ["class b; endclass class c extends b(default); endclass", "unsupported syntax: ExtendsClause"],
  ["class c; virtual function :final void f(); endfunction endclass", "unsupported syntax: FunctionPrototype"],
  ["class c; local typedef int t; endclass", "unsupported syntax: ClassPropertyDeclaration"],
  ["class c; ; endclass", "unsupported syntax: EmptyMember"],
  ["module m; var type(a) b; endmodule", "unsupported syntax: TypeReference"],
  ["class c; constraint :initial k { x > 0; } endclass", "unsupported syntax: ConstraintDeclaration"],
  ["class c; extern constraint :final k; endclass", "unsupported syntax: ConstraintPrototype"],
  ["module m; initial randsequence() main: {}; endsequence endmodule", "unsupported syntax: RandSequenceStatement"],
  ["class c; function void f(); ok = obj.randomize() with (a.b) { x > 0; }; endfunction endclass",
    "unsupported syntax: ScopedName"],
  ["class c; constraint k { foreach (q[, j]) q[j] > 0; } endclass", "unsupported syntax: EmptyIdentifierName"],
  ["class c; function void f(); x = local::p::y; endfunction endclass", "unsupported syntax: ScopedName"],
  ["module m; property p; (not a, x = 1); endproperty endmodule", "unsupported syntax: ParenthesizedPropertyExpr"],
] as const) {
  fails(source + "\n", message);
}
console.log("ok");

## VLGRD-12 · Corner cases the kinds hold

In [ ]:
unit = read(`// é: positions count characters, not bytes
module m #(parameter type T, parameter W) ();
logic a [int]; logic b [t]; automatic int c;
function static int f(); endfunction
if (1) begin end
u #(.N()) i(); u j();
initial begin
    for (i = 0, j = 1; i < 2; i++) ;
    @(posedge a or b iff c) ;
    @(e.triggered) ;
    x = a.b[1].c + a.b.c + t::a;
    x = '{a, b} + q.sum() + int'(1) + signed'(1) + a.b();
    x = $bits(int) + $bits(logic [1:0]) + f()[0] + f(a).b + f(.a(), .b(c));
    x: begin end
    for (int k; k < 2; k++) ;
    assert (a);
    x = """multi
line""";
end
if (1) g: begin end
endmodule
`);
module = unit.items[1];
assert(unit.items[0].text === " é: positions count characters, not bytes");
assert(module.parameters[0].assignments[0].type === null && module.parameters[1].assignments[0].value === null);
assert(module.items[0].declarators[0].dimensions[0].type.keyword === "int");
assert(module.items[1].declarators[0].dimensions[0].size.name.spelling === "t");
assert(module.items[2].lifetime === "automatic" && module.items[3].lifetime === "static");
assert(module.items[5].parameters[0].value === null && module.items[6].instances[0].connections.length === 0);
body = module.items[7].body.items;
assert(body[0].initializers.length === 2 && body[1].timing.events[1].condition.name.spelling === "c");
equal(body[5].value.left.left.left.left.arguments.map(kind), ["IntegerAtomType"]);
assert(body[6].name.spelling === "x" && body[7].initializers[0].declarators[0].value === null);
assert(module.items[8].consequence.name.spelling === "g");
assert(body[8].pass_action === null && body[8].fail_action === null);
const string = body[9].value;
assert(string.triple && string.text === "multi\nline");
raises(ParseError, () => SystemVerilog2017.parse('module m; initial x = """s"""; endmodule\n'),
  "StringLiteral.triple needs SystemVerilog-2023, but this is SystemVerilog-2017");
assert(F.same(read(SystemVerilog2023.print(unit)), unit));
fails("// é\nmodule m; wire = ; endmodule\n", "line 2, column 16");
console.log("ok");

## VLGRD-13 · Reading for the other implementation: a request in, a tree or an error out, as JSON

In [ ]:
const source = "module m; wire a; endmodule\n";
const parsed = Reader.parse(source, 2023, "SystemVerilog", [], []);
assert(F.same(parsed, read(source)) && parsed.items[0] instanceof S.ModuleDeclaration);
raises(ParseError, () => Reader.parse("module m;\n wire = ;\nendmodule\n", 2023, "SystemVerilog", [], []),
  "line 2, column 7: expected declarator");
raises(ParseError, () => Reader.parse("package p; endpackage\n", 2005, "Verilog", [], []),
  "PackageDeclaration is not Verilog, but this is Verilog-2005");
assert(Reader.python().endsWith("python3/.venv/bin/python") || process.env["MBSE_PROGRAMS_PYTHON"] !== undefined);
// Reading runs Python, which must be there: MBSE_PROGRAMS_PYTHON names the interpreter.
const saved = process.env["MBSE_PROGRAMS_PYTHON"];
process.env["MBSE_PROGRAMS_PYTHON"] = "/nonexistent/python";
try {
  equal(Reader.python(), "/nonexistent/python");
  raises(Error, () => read("module m; endmodule\n"), "reading Verilog runs /nonexistent/python with mbse-programs and pyslang");
} finally {
  if (saved === undefined) delete process.env["MBSE_PROGRAMS_PYTHON"];
  else process.env["MBSE_PROGRAMS_PYTHON"] = saved;
}
console.log("ok");

## VLGRD-14 · Classes, their properties and methods, and objects

In [ ]:
unit = read(`typedef class fwd;
typedef plain;
typedef interface class ic;
virtual class base #(type T = int, int N = 2) extends parent #(T) implements i1, i2;
  rand bit [3:0] a; randc int b; local static const int c = 1; protected int d;
  virtual bus_if.mp vif; virtual interface bus_if #(4) v2; c #(8, .T(int)) other;
  extern virtual function void f(int x);
  pure virtual task t();
  extern protected static function int g();
  function new(int x = 0); super.new(x); this.a = x; x = new; x = new y; d = new[4](d); d = new[2];
    x = null; x = p::q::r; x = c#(1)::new(3); x = c#(1)::s; super.g(); endfunction
  local virtual task run(); endtask
endclass : base
class derived extends base #(byte, 3)(1, .x(2)); endclass
interface class i1 extends j, k; pure virtual function void h(); endclass
`);
const [forward, plain, forwardInterface, baseClass, derived, interfaceClass] = unit.items;
assert(plain.keyword === null && plain.name.spelling === "plain");
assert(forward.keyword === "class" && forwardInterface.keyword === "interface class" && forward.name.spelling === "fwd");
equal([baseClass.virtual, baseClass.interface, baseClass.labeled, baseClass.name.spelling], [true, false, true, "base"]);
equal(baseClass.parameters.map(kind), ["TypeParameterDeclaration", "ParameterDeclaration"]);
assert(baseClass.parameters[0].keyword === null && baseClass.parameters[1].type.keyword === "int");
assert(baseClass.base.name instanceof S.ParameterizedName && baseClass.base.name.parameters[0].name.spelling === "T");
equal(baseClass.interfaces.map((i: any) => i.name.spelling), ["i1", "i2"]);
assert(baseClass.arguments.length === 0);
const [rnd, rndc, hidden, prot, vif, v2, other, ext, pure, proto, constructor, run] = baseClass.items;
equal([rnd.random, rndc.random, hidden.visibility, hidden.lifetime, hidden.const, prot.visibility], ["rand", "randc", "local", "static", true,
  "protected"]);
equal([vif.type.interface.spelling, vif.type.modport.spelling, vif.type.interface_keyword], ["bus_if", "mp", false]);
assert(v2.type.interface_keyword && v2.type.parameters[0].spelling === "4" && v2.type.modport === null);
equal(other.type.name.parameters.map(kind), ["IntegerLiteral", "NamedConnection"]);
equal([ext.extern, ext.virtual, ext.pure, ext.body], [true, true, false, []]);
equal([pure.pure, pure.virtual], [true, true]);
equal([proto.extern, proto.visibility, proto.static, proto.lifetime], [true, "protected", true, null]);
assert(constructor.name.spelling === "new" && constructor.type === null);
equal([run.visibility, run.virtual], ["local", true]);
const bodyValues = constructor.body.map((s: any) => s instanceof S.AssignmentStatement ? s.value : s.expression);
assert(bodyValues[0] instanceof S.NewExpression && bodyValues[0].scope instanceof S.SuperExpression);
assert(bodyValues[1] instanceof S.NameExpression && constructor.body[1].target.value instanceof S.ThisExpression);
assert(bodyValues[2].scope === null && bodyValues[2].arguments.length === 0 && bodyValues[3].value.name.spelling === "y");
assert(bodyValues[4].size.spelling === "4" && bodyValues[4].value.name.spelling === "d" && bodyValues[5].value === null);
assert(bodyValues[6] instanceof S.NullLiteral);
const nested = bodyValues[7].name;
equal([nested.scope.spelling, nested.name.scope.spelling, nested.name.name.spelling], ["p", "q", "r"]);
assert(bodyValues[8].scope.name.spelling === "c" && bodyValues[8].arguments[0].spelling === "3");
assert(bodyValues[9].name.scope instanceof S.ParameterizedName && bodyValues[9].name.name.spelling === "s");
assert(bodyValues[10].callee.value instanceof S.SuperExpression);
assert(derived.base.name.parameters[0].keyword === "byte" && derived.arguments.length === 2);
assert(interfaceClass.interface);
equal(interfaceClass.interfaces.map((i: any) => i.name.spelling), ["j", "k"]);
assert(F.same(read(SystemVerilog2023.print(unit)), unit));
console.log("ok");

## VLGRD-15 · Constraints, randomization with inline constraints, array methods with `with`, `randcase` and `void'`

In [ ]:
unit = read(`class c;
  rand int x, y; rand int q[];
  constraint k { // first
    x > 0; soft y < 10; x -> { y == 1; } x -> y == 2; if (x) y > 2; else { y < 3; } foreach (q[i]) q[i] < 5;
    solve x before y, q; disable soft y; unique { x, y, [1:2] }; x dist { 1 := 2, [3:4] :/ 5, 7, default :/ 1 }; }
  static constraint s { }
  extern constraint e;
  pure constraint p;
  static constraint plain;
  function void f();
    ok = this.randomize() with { x < local::y; }; ok = obj.randomize(x, y) with (x) { x > 0; };
    ok = std::randomize(a, b) with { a < b; }; ok = obj.randomize() with () { /* none */ };
    r = q.find(item) with (item > 1); q.sort with (item.x);
    randcase 1: a = 1; x + 1: begin end endcase
    void'(obj.randomize());
  endfunction
endclass
constraint c::e { x != y; }
`);
const [randomClass, constraintOutside] = unit.items;
const [kConstraint, empty, prototypeExtern, prototypePure, prototypePlain] = randomClass.items.slice(2, 7);
assert(kConstraint.name.spelling === "k" && !kConstraint.static && kConstraint.items[0].text === " first" && kConstraint.items[0].trailing);
const constraints = kConstraint.items.slice(1);
equal(constraints.map(kind), ["ExpressionConstraint", "ExpressionConstraint", "ImplicationConstraint", "ImplicationConstraint",
  "ConditionalConstraint", "ForeachConstraint", "SolveBeforeConstraint", "DisableSoftConstraint", "UniqueConstraint",
  "ExpressionConstraint"]);
assert(!constraints[0].soft && constraints[1].soft);
assert(constraints[2].body instanceof S.ConstraintBlock && constraints[3].body instanceof S.ExpressionConstraint);
assert(constraints[4].consequence instanceof S.ExpressionConstraint && constraints[4].alternative instanceof S.ConstraintBlock);
equal(constraints[5].variables.map((v: any) => v.spelling), ["i"]);
assert(constraints[5].array.name.spelling === "q");
equal(constraints[6].solve.map((e: any) => e.name.spelling), ["x"]);
equal(constraints[6].before.map((e: any) => e.name.spelling), ["y", "q"]);
assert(constraints[7].target.name.spelling === "y" && constraints[8].set[2] instanceof S.ValueRange);
const dist = constraints[9].expression;
assert(dist instanceof S.DistExpression && (dist.value as any).name.spelling === "x");
equal(dist.items.map((i: any) => [i.value === null ? null : kind(i.value), i.operator]),
  [["IntegerLiteral", ":="], ["ValueRange", ":/"], ["IntegerLiteral", null], [null, ":/"]]);
assert(empty.static && empty.items.length === 0);
equal([prototypeExtern.qualifier, prototypePure.qualifier, prototypePlain.qualifier, prototypePlain.static],
  ["extern", "pure", null, true]);
const methodBody = randomClass.items[7].body;
const calls = methodBody.slice(0, 5).map((s: any) => s.value);
assert(calls[0] instanceof S.RandomizeWithExpression && !calls[0].restricted);
assert(calls[0].items[0].expression.right.name instanceof S.LocalName);
assert(calls[1].restricted && calls[1].call.arguments.length === 2);
equal(calls[1].variables.map((v: any) => v.spelling), ["x"]);
assert(calls[2].call.callee.name.scope.spelling === "std");
assert(calls[3].restricted && calls[3].variables.length === 0 && calls[3].items[0].block);
assert(calls[4] instanceof S.ArrayMethodWithExpression && calls[4].expression.operator === ">");
assert(methodBody[5].expression instanceof S.ArrayMethodWithExpression && methodBody[5].expression.call instanceof S.MemberExpression);
equal(methodBody[6].items.map((i: any) => i.weight instanceof S.IntegerLiteral ? i.weight.spelling : i.weight.operator), ["1", "+"]);
assert(methodBody[7].expression.type.keyword === "void" && methodBody[7].expression.value instanceof S.CallExpression);
assert(constraintOutside.name.scope.spelling === "c" && constraintOutside.items[0].expression.operator === "!=");
assert(F.same(read(SystemVerilog2023.print(unit)), unit));
console.log("ok");

## VLGRD-16 · Assertions: properties, sequences, clocking blocks, cycle delays, `let` and labels

In [ ]:
const assertions: any = read(`module m;
  default clocking cb @(posedge clk); default input #1step output #0; input #1 a, b; output negedge c; inout d = top.x;
    input #2 output #3 e; endclocking : cb
  global clocking gc @(posedge fast); endclocking
  default clocking @(negedge clk); property inner; if (a) b; endproperty endclocking
  default clocking cb;
  default disable iff (rst);
  sequence s1(x, int y = 1, local input logic z); int v; @(posedge clk) (a, v = x) ##1 b[*2] ##[1:3] c[->1] ##[*] d[=2:3]
    ##[+] e[*] ##1 f[+] ##(y) g; endsequence : s1
  sequence s2; ##1 a ##2 (b ##1 c)[*3] and first_match(d ##1 e, v++) or f intersect g within h throughout i; endsequence
  property p1(sequence q, untyped u); disable iff (rst) q |-> not (a until b) and s_eventually [1:$] c; endproperty
  property p2; if (a) strong(b ##1 c) else weak(d) implies nexttime [2] e iff always [1:3] f; endproperty
  property p3; accept_on (x) a s_until b or sync_reject_on (y) c until_with d; endproperty
  property p4; case (m) 0, 1: a |=> b; default: (c #=# d); endcase; endproperty
  a1: assert property (@(posedge clk) disable iff (rst) a |=> b) else $error("x");
  cover sequence (s1(1));
  restrict property (a ##1 @(negedge clk) b);
  assume property (s_nexttime a or @(negedge clk) b |-> c);
  assert #0 (x);
  let max(a, b = 1) = a > b ? a : b;
  initial begin ##2; ##1 cb.c <= 1; cb.c <= ##2 a; chk: assert (x); a2: assert property (p1(s1, 1));
    expect (@(posedge clk) a ##1 b) else $error; lab: x = 1; end
endmodule
`).items[0];
const [cb, gc, anonymous, defaultReference, disable, s1, s2, p1, p2, p3, p4, a1, coverItem, restrictItem, assumeItem,
  deferred, letItem, initialItem] = assertions.items;
equal([cb.scope, cb.name.spelling, cb.labeled, gc.scope, anonymous.name], ["default", "cb", true, "global", null]);
assert(anonymous.items[0] instanceof S.PropertyDeclaration && anonymous.items[0].spec.property.alternative === null);
const [skews, inputs, outputs, inouts, both] = cb.items;
assert(skews.input.delay.value.spelling === "1step" && skews.output.delay.value.spelling === "0");
equal([inputs.direction, inputs.signals.map((s: any) => s.name.spelling), inputs.input_skew.delay.value.spelling],
  ["input", ["a", "b"], "1"]);
equal([outputs.direction, outputs.output_skew.edge, outputs.output_skew.delay], ["output", "negedge", null]);
assert(inouts.direction === "inout" && inouts.signals[0].value.member.spelling === "x");
assert(both.direction === "input output" && both.output_skew.delay.value.spelling === "3");
assert(defaultReference.name.spelling === "cb" && disable.condition instanceof S.ParenthesizedExpression);
equal(s1.ports.map((p: any) => [p.local, p.direction, p.type && kind(p.type), p.value && p.value.spelling]),
  [[false, null, null, null], [false, null, "IntegerAtomType", "1"], [true, "input", "IntegerVectorType", null]]);
assert(s1.labeled && s1.variables[0].declarators[0].name.spelling === "v");
const clocked = s1.sequence;
assert(clocked instanceof S.ClockedSequence && clocked.sequence instanceof S.DelaySequence);
const chain: any = clocked.sequence;
assert(chain.first instanceof S.ParenthesizedSequence && chain.first.items[0].operator === "=");
equal(chain.steps.map((s: any) => kind(s.delay)), ["IntegerLiteral", "CycleRange", "CycleRange", "CycleRange", "IntegerLiteral",
  "ParenthesizedExpression"]);
equal(chain.steps.slice(2, 4).map((s: any) => [s.delay.low.spelling, kind(s.delay.high)]), [["0", "DollarExpression"],
  ["1", "DollarExpression"]]);
const repetitions = chain.steps.slice(0, 5).map((s: any) => s.sequence);
equal(repetitions.map((r: any) => [r.operator, kind(r.count)]), [["*", "IntegerLiteral"], ["->", "IntegerLiteral"],
  ["=", "CycleRange"], ["*", "CycleRange"], ["*", "CycleRange"]]);
equal([repetitions[3].count.low.spelling, repetitions[4].count.low.spelling], ["0", "1"]);
assert(s2.sequence.operator === "or" && s2.sequence.left.operator === "and" && s2.sequence.left.left.first === null);
assert(s2.sequence.left.right instanceof S.FirstMatchSequence && s2.sequence.right.right.right.operator === "throughout");
equal([p1.ports[0].type.keyword, p1.ports[1].type.keyword], ["sequence", "untyped"]);
const implication = p1.spec.property;
assert(p1.spec.disable.name.spelling === "rst");
equal([implication.operator, implication.consequent.operator], ["|->", "and"]);
assert(implication.consequent.left.operator === "not" && implication.consequent.left.operand instanceof S.ParenthesizedProperty);
assert(implication.consequent.right.range.high instanceof S.DollarExpression);
const ifProperty = p2.spec.property;
assert(ifProperty instanceof S.ConditionalProperty && (ifProperty.consequence as any).keyword === "strong");
const alternative: any = ifProperty.alternative; // `implies` binds looser than `iff`
assert(alternative.operator === "implies" && alternative.right.operator === "iff" && alternative.right.left.range.spelling === "2");
const abort = p3.spec.property; // `or` binds tighter than `s_until`
assert(abort.keyword === "accept_on" && abort.operand.operator === "s_until" && abort.operand.right.right.keyword === "sync_reject_on");
const caseProperty = p4.spec.property;
equal(caseProperty.items.map((i: any) => i.expressions.length), [2, 0]);
assert(caseProperty.items[1].body instanceof S.ParenthesizedProperty);
assert(a1.label.spelling === "a1" && a1.assertion.spec.clock.events[0].edge === "posedge");
assert(a1.assertion.fail_action.expression.name === "$error" && a1.assertion.pass_action === null);
assert(coverItem.assertion.sequence && coverItem.assertion.keyword === "cover" && restrictItem.assertion.keyword === "restrict");
assert(restrictItem.assertion.spec.property.steps[0].sequence instanceof S.ClockedSequence);
assert(assumeItem.assertion.spec.property.right instanceof S.ClockedProperty);
assert(deferred.assertion instanceof S.ImmediateAssertion && deferred.label === null);
equal(letItem.ports.map((p: any) => p.name.spelling), ["a", "b"]);
assert(letItem.value instanceof S.ConditionalExpression);
const steps = initialItem.body.items;
assert(steps[0].timing instanceof S.CycleDelay && steps[0].body === null && steps[1].body.operator === "<=");
assert(steps[2].timing instanceof S.CycleDelay);
assert(steps[3] instanceof S.LabeledStatement && steps[3].statement instanceof S.ImmediateAssertion);
assert(steps[4].label.spelling === "a2" && steps[4].statement instanceof S.ConcurrentAssertion);
assert(steps[5] instanceof S.ExpectStatement && steps[5].fail_action !== null);
assert(steps[6].label.spelling === "lab" && steps[6].statement instanceof S.AssignmentStatement);
assert(F.same((read(SystemVerilog2023.print(assertions)) as any).items[0], assertions));
console.log("ok");